5) Clustering-Map-Refine 요약

In [ ]:
# [목적] PyMuPDFLoader로 PDF를 페이지별 문서로 불러와 클러스터링 요약의 원문을 준비합니다.
# PDF의 각 페이지를 LangChain Document 객체로 변환하고 전체 페이지 수를 확인합니다.
# 준비된 docs는 하나의 텍스트로 합친 뒤 분할·임베딩하는 다음 단계에 사용합니다.
from langchain_community.document_loaders import PyMuPDFLoader

# PyMuPDFLoader는 PDF 본문과 페이지 정보를 Document 객체에 담아 반환합니다.
loader = PyMuPDFLoader("data/SPRI_AI_Brief_2023년12월호_F.pdf")
docs = loader.load()

len(docs)

In [ ]:
# [목적] 페이지별 PDF 본문을 하나의 문자열로 합쳐 텍스트 분할 입력을 만듭니다.
# 각 Document의 page_content를 빈 줄로 연결해 페이지 사이의 구분을 남깁니다.
# 합친 텍스트의 길이를 확인하고 다음 셀에서 일정한 크기의 청크로 나눕니다.
texts = "\n\n".join([doc.page_content for doc in docs])
len(texts)

In [ ]:
# [목적] 긴 PDF 본문을 임베딩과 클러스터링에 적합한 작은 텍스트 청크로 분할합니다.
# RecursiveCharacterTextSplitter가 문맥 경계를 고려해 500자 단위로 나누고 앞뒤 청크를 100자씩 겹칩니다.
# 겹치는 내용은 청크 경계에서 중요한 문맥이 끊기는 현상을 줄여 줍니다.
from langchain_text_splitters import RecursiveCharacterTextSplitter

# chunk_size는 청크 목표 크기이고 chunk_overlap은 이웃 청크와 공유할 문자 수입니다.
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
split_docs = text_splitter.split_text(texts)

In [ ]:
# [목적] 분할된 텍스트 청크의 개수를 확인합니다.
# split_docs의 길이를 출력해 임베딩 API에 전달될 입력 규모를 점검합니다.
# 이 개수만큼 의미 벡터가 생성되고 각각 클러스터에 배정됩니다.
len(split_docs)

In [ ]:
# [목적] 각 텍스트 청크를 의미를 수치로 표현한 임베딩 벡터로 변환합니다.
# OpenAIEmbeddings가 split_docs를 벡터화하고 결과를 원래 청크 순서대로 vectors에 저장합니다.
# 이 벡터들은 내용이 비슷한 청크를 K-Means로 묶는 기준으로 사용합니다.
from langchain_openai import OpenAIEmbeddings

# OpenAIEmbeddings는 환경 변수의 OpenAI API 키를 사용해 텍스트의 의미 벡터를 생성합니다.
embeddings = OpenAIEmbeddings()

# embed_documents는 여러 문자열을 한 번에 변환해 청크별 벡터 목록을 반환합니다.
vectors = embeddings.embed_documents(split_docs)

In [ ]:
# [목적] 임베딩 벡터를 의미가 비슷한 10개 그룹으로 군집화합니다.
# K-Means가 각 벡터를 가장 가까운 중심점에 배정하고 클러스터 중심을 계산합니다.
# 각 중심에 가까운 대표 청크를 고르면 전체 문서를 적은 수의 핵심 청크로 압축할 수 있습니다.
from sklearn.cluster import KMeans

num_clusters = 10

# random_state를 고정하면 같은 입력에서 초기 중심 선택과 군집 결과를 재현하기 쉽습니다.
kmeans = KMeans(n_clusters=num_clusters, random_state=123).fit(vectors)

In [ ]:
# [목적] 각 텍스트 청크가 배정된 클러스터 번호를 확인합니다.
# labels_에는 vectors와 같은 순서로 0부터 9까지의 군집 번호가 저장됩니다.
# 이 번호는 시각화에서 색을 나누고 대표 청크를 선택할 때 사용합니다.
kmeans.labels_

In [ ]:
# [목적] 고차원 임베딩과 K-Means 군집 결과를 2차원 산점도로 시각화합니다.
# t-SNE로 벡터 차원을 줄인 뒤 클러스터 번호별로 색을 달리해 청크의 분포를 표시합니다.
# 군집이 어떻게 나뉘었는지 눈으로 확인해 대표 청크 선택 결과를 이해하는 데 사용합니다.
import warnings

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from sklearn.manifold import TSNE

# 시각화 과정에서 발생하는 부가 경고를 숨겨 결과 그래프에 집중합니다.
warnings.filterwarnings("ignore")

# t-SNE는 가까운 벡터끼리 이웃하도록 고차원 데이터를 2차원 좌표로 축소합니다.
tsne = TSNE(n_components=2, random_state=42)
reduced_data_tsne = tsne.fit_transform(np.array(vectors))

sns.set_style("white")  # Seaborn 스타일 설정

plt.figure(figsize=(10, 8))  # 축소된 데이터 플롯
sns.scatterplot(
    x=reduced_data_tsne[:, 0],
    y=reduced_data_tsne[:, 1],
    hue=kmeans.labels_,
    palette="deep",
    s=100,
)

plt.xlabel("Dimension 1", fontsize=12)
plt.ylabel("Dimension 2", fontsize=12)
plt.title("Clustered Embeddings", fontsize=16)
plt.legend(title="Cluster", title_fontsize=12)

plt.gcf().patch.set_facecolor("white")  # 배경색 설정

plt.tight_layout()
plt.show()

In [ ]:
# [목적] 각 클러스터 중심에 가장 가까운 청크 하나를 대표 문서로 선택합니다.
# 모든 벡터와 각 중심점 사이의 거리를 계산하고 가장 짧은 거리의 청크 인덱스를 저장합니다.
# 선택된 10개 청크만 요약하면 원문 전체의 다양한 주제를 비교적 고르게 반영할 수 있습니다.
import numpy as np

closest_indices = []  # 클러스터별 대표 청크의 위치를 저장합니다.

for i in range(num_clusters):  # 각 클러스터 중심을 하나씩 확인합니다.

    # linalg.norm은 각 벡터와 현재 중심점 사이의 직선거리를 계산합니다.
    distances = np.linalg.norm(vectors - kmeans.cluster_centers_[i], axis=1)

    # argmin은 거리 목록에서 가장 작은 값의 위치를 반환합니다.
    closest_index = np.argmin(distances)

    # 해당 인덱스를 가장 가까운 인덱스 리스트에 추가
    closest_indices.append(closest_index)

In [ ]:
# [목적] 클러스터별 대표 청크로 선택된 인덱스 목록을 확인합니다.
# 각 숫자는 split_docs에서 해당 대표 텍스트가 있는 위치를 뜻합니다.
# 다음 셀에서 이 위치들을 정렬해 원문에 등장한 순서로 복원합니다.
closest_indices

In [ ]:
# [목적] 대표 청크 인덱스를 원문에 등장한 순서대로 정렬합니다.
# 클러스터 번호순이 아니라 PDF 본문의 흐름순으로 배치해 요약이 자연스럽게 이어지도록 합니다.
# 정렬된 selected_indices는 실제 대표 Document를 만드는 데 사용합니다.
selected_indices = sorted(closest_indices)
selected_indices

In [ ]:
# [목적] 선택한 대표 텍스트를 Map-Refine 체인이 처리할 LangChain Document로 변환합니다.
# 정렬된 인덱스로 split_docs의 본문을 꺼내 각각 Document의 page_content에 담습니다.
# 생성된 selected_docs만 요약해 전체 청크를 사용할 때보다 모델 호출량을 줄입니다.
from langchain_core.documents import Document

# Document는 본문과 메타데이터를 LangChain 구성요소 사이에서 전달하는 표준 객체입니다.
selected_docs = [
    Document(page_content=split_docs[doc])
    for doc in selected_indices
]

selected_docs

In [ ]:
# [목적] 대표 청크를 각각 요약하고 순서대로 보완하는 Map-Refine 체인을 정의합니다.
# Map 단계에서 청크별 초벌 요약을 만든 뒤 Refine 단계에서 앞 요약에 다음 내용을 누적 반영합니다.
# 클러스터링으로 선별한 핵심 내용들을 하나의 자연스러운 한국어 요약으로 통합합니다.
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import chain
from langchain_openai import ChatOpenAI
from langsmith import Client

# Client는 LangSmith Hub에 저장된 Map·Refine 프롬프트를 불러오는 연결 객체입니다.
client = Client()

# @chain은 일반 Python 함수를 invoke로 실행할 수 있는 LangChain Runnable로 변환합니다.
@chain
def map_refine_chain(docs):

    map_summary = client.pull_prompt(
        "teddynote/map-summary-prompt",
        dangerously_pull_public_prompt=True
    )

    map_chain = (
        map_summary
        | ChatOpenAI(
            model_name="gpt-4o-mini",
            temperature=0,
        )
        | StrOutputParser()
    )

    # 각 Document의 본문과 출력 언어를 Map 프롬프트의 입력 형식으로 맞춥니다.
    input_doc = [
        {"documents": doc.page_content, "language": "Korean"}
        for doc in docs
    ]

    # batch로 모든 대표 청크를 처리해 입력 순서가 유지된 초벌 요약 목록을 만듭니다.
    doc_summaries = map_chain.batch(input_doc)

    refine_prompt = client.pull_prompt(
        "teddynote/refine-prompt",
        dangerously_pull_public_prompt=True
    )

    refine_chain = (
        refine_prompt
        | ChatOpenAI(
            model_name="gpt-4o-mini",
            temperature=0,
        )
        | StrOutputParser()
    )

    # 첫 요약을 시작값으로 두고 이후 요약을 하나씩 반영해 누적 결과를 갱신합니다.
    previous_summary = doc_summaries[0]

    for current_summary in doc_summaries[1:]:
        previous_summary = refine_chain.invoke(
            {
                "previous_summary": previous_summary,
                "current_summary": current_summary,
                "language": "Korean",
            }
        )

    return previous_summary

In [ ]:
# [목적] 선별된 대표 문서에 Map-Refine 체인을 실행해 최종 요약을 생성합니다.
# selected_docs가 Map 단계와 반복 Refine 단계를 차례로 거쳐 하나의 결과로 합쳐집니다.
# 완성된 한국어 요약은 refined_summary에 저장해 다음 셀에서 확인합니다.
refined_summary = map_refine_chain.invoke(selected_docs)

In [ ]:
# [목적] Clustering-Map-Refine 과정을 거쳐 완성된 최종 요약을 출력합니다.
# 앞 셀에서 반환된 refined_summary를 화면에 표시해 대표 청크의 내용이 잘 통합되었는지 확인합니다.
# 결과는 필요에 따라 파일 저장이나 후속 분석의 입력으로 사용할 수 있습니다.
print(refined_summary)